Packages

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Paths to Result Summaries

In [ ]:
summary_fed_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results")
summary_ecb_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/ECB Side/results")
# Summary FED. 
summary_fed_statements_pressconf = pd.read_csv(summary_fed_path / "summary__fed_statements_presconf.csv")
summary_fed_speeches = pd.read_csv(summary_fed_path / "summary__fed_speeches.csv" )
# Summary ECB
summary_ecb_statements_pressconf = pd.read_csv(summary_ecb_path / "summary__ECB_statements_presconf.csv")
summary_ecb_speeches = pd.read_csv(summary_ecb_path / "summary__ECB_speeches.csv")


In [11]:
summary_ecb_statements_pressconf.head(20)

,model,prompt,n,parsed,errors,accuracy,macro_f1,in_tok,out_tok,cost_usd
0,claude-haiku,p1_analyst_direct,2044,2043,0,0.7200,0.7011,525434,10220,0.5765
1,claude-haiku,p2_analyst_rubric,2044,2043,0,0.6955,0.6835,917882,10220,0.9690
2,claude-haiku,p3_analyst_rules,2044,2043,0,0.6637,0.6218,576534,143327,1.2932
3,claude-haiku,p4_analyst_json,2044,2043,0,0.6985,0.6633,410970,23676,0.5293
4,claude-haiku,p5_analyst_fewshot,2044,2043,0,0.6897,0.6767,619458,10220,0.6706
5,claude-sonnet,p1_analyst_direct,2044,2043,0,0.7014,0.6854,527478,10369,1.7380
6,claude-sonnet,p2_analyst_rubric,2044,2043,0,0.6907,0.6675,919926,10220,2.9131
7,claude-sonnet,p3_analyst_rules,2044,2043,0,0.6804,0.6553,578578,36703,2.2863
8,claude-sonnet,p4_analyst_json,2044,2043,0,0.7200,0.7070,413014,18396,1.5150
9,claude-sonnet,p5_analyst_fewshot,2044,2043,0,0.6769,0.6602,621502,10220,2.0178


In [10]:
print(summary_ecb_statements_pressconf.columns)

Index(['model', 'prompt', 'n', 'parsed', 'errors', 'accuracy', 'macro_f1',
       'in_tok', 'out_tok', 'cost_usd'],
      dtype='str')


In [7]:
summary_ecb_speeches.head(2)

,model,prompt,n,parsed,errors,accuracy,macro_f1,in_tok,out_tok,cost_usd
0,claude-haiku,p1_analyst_direct,1000,1000,0,0.683,0.6839,256644,5000,0.2816
1,claude-haiku,p2_analyst_rubric,1000,1000,0,0.649,0.6518,448644,5000,0.4736


In [12]:
# ============================================================
# FED vs ECB — model × prompt comparison on macro-F1
# Paste below your loading cell (uses your existing variables).
# ============================================================
import numpy as np
import pandas as pd

try:
    display  # available in notebooks
except NameError:
    display = print

pd.set_option("display.float_format", lambda v: f"{v:.3f}")

runs = pd.concat(
    [
        summary_fed_statements_pressconf.assign(bank="FED", doc_type="statements_pressconf"),
        summary_fed_speeches.assign(bank="FED", doc_type="speeches"),
        summary_ecb_statements_pressconf.assign(bank="ECB", doc_type="statements_pressconf"),
        summary_ecb_speeches.assign(bank="ECB", doc_type="speeches"),
    ],
    ignore_index=True,
)
runs["dataset"] = runs["bank"] + "_" + runs["doc_type"]

# ------------------------------------------------------------
# A · Raw picture — mean macro-F1 per model, per bank
#     (averaged over the 5 prompts and both doc types)
#     NB: ECB_minus_FED here mixes model skill with dataset
#     difficulty — table C fixes that.
# ------------------------------------------------------------
model_by_bank = (
    runs.pivot_table(index="model", columns="bank", values="macro_f1", aggfunc="mean")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("A · mean macro-F1 per model × bank")
display(model_by_bank)

# finer grain: one column per dataset
model_by_dataset = runs.pivot_table(
    index="model", columns="dataset", values="macro_f1", aggfunc="mean"
)
print("A2 · mean macro-F1 per model × dataset")
display(model_by_dataset)

# ------------------------------------------------------------
# B · Prompt lift — your idea:
#     each run minus that model's OWN average on the same
#     dataset. Positive = this prompt pushes the model above
#     its usual level; comparable across models because each
#     model is measured against itself.
# ------------------------------------------------------------
runs["prompt_lift"] = runs["macro_f1"] - runs.groupby(["dataset", "model"])[
    "macro_f1"
].transform("mean")

prompt_lift_by_bank = runs.pivot_table(
    index="prompt", columns="bank", values="prompt_lift", aggfunc="mean"
)
print("B · average prompt lift, net of model strength")
display(prompt_lift_by_bank)

# per-model detail (heatmap-ready): which prompt works for WHICH model
prompt_lift_detail = runs.pivot_table(
    index="model", columns=["bank", "prompt"], values="prompt_lift", aggfunc="mean"
)
print("B2 · prompt lift per model (columns: bank × prompt)")
display(prompt_lift_detail)

# ------------------------------------------------------------
# C · Relative model skill — the same centring trick, flipped:
#     each run minus the DATASET average over all models and
#     prompts. This removes dataset difficulty, so
#     ECB_minus_FED > 0 really means "relatively stronger on
#     ECB than the field", not "ECB was easier".
# ------------------------------------------------------------
runs["rel_skill"] = runs["macro_f1"] - runs.groupby("dataset")["macro_f1"].transform("mean")

rel_skill_by_bank = (
    runs.pivot_table(index="model", columns="bank", values="rel_skill", aggfunc="mean")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("C · relative skill (dataset-demeaned macro-F1)")
display(rel_skill_by_bank)

# ------------------------------------------------------------
# Extra · best prompt per model & bank (on average macro-F1)
# ------------------------------------------------------------
mp = runs.groupby(["bank", "model", "prompt"], as_index=False)["macro_f1"].mean()
best_prompt = (
    mp.loc[mp.groupby(["bank", "model"])["macro_f1"].idxmax()]
      .sort_values(["bank", "macro_f1"], ascending=[True, False])
      .reset_index(drop=True)
)
print("Best prompt per model × bank")
display(best_prompt)

# ------------------------------------------------------------
# Notes / variants
# ------------------------------------------------------------
# 1) Robustness: if one prompt tanks a model (e.g. JSON parsing
#    failures), the mean gets dragged down. Re-run tables A and C
#    with aggfunc="median" to check the ranking is stable.
#
# 2) Sample-size weighting: statements (~2044 docs) and speeches
#    (~1000) currently count equally in the bank-level means.
#    Document-weighted alternative:
#    (runs.groupby(["model", "bank"])
#         .apply(lambda g: np.average(g["macro_f1"], weights=g["n"]))
#         .unstack("bank"))

A · mean macro-F1 per model × bank


bank,ECB,FED,ECB_minus_FED
model,,,
deepseek-reasoner,0.706,0.653,0.053
mistral-large,0.668,0.624,0.044
deepseek-chat,0.593,0.550,0.044
claude-sonnet,0.677,0.635,0.042
claude-haiku,0.660,0.629,0.031
gpt-5.1,0.665,0.646,0.019
mistral-small,0.610,0.595,0.015
gpt-5-mini,0.634,0.635,-0.001


A2 · mean macro-F1 per model × dataset


dataset,ECB_speeches,ECB_statements_pressconf,FED_speeches,FED_statements_pressconf
model,,,,
claude-haiku,0.651,0.669,0.623,0.636
claude-sonnet,0.678,0.675,0.660,0.610
deepseek-chat,0.563,0.623,0.561,0.538
deepseek-reasoner,0.713,0.699,0.665,0.641
gpt-5-mini,0.604,0.663,0.630,0.640
gpt-5.1,0.636,0.695,0.648,0.644
mistral-large,0.655,0.681,0.627,0.620
mistral-small,0.600,0.619,0.593,0.596


B · average prompt lift, net of model strength


...

bank                            ECB                                     \
prompt            p1_analyst_direct p2_analyst_rubric p3_analyst_rules   
model                                                                    
claude-haiku                  0.032             0.007           -0.048   
claude-sonnet                 0.017            -0.012           -0.017   
deepseek-chat                 0.035            -0.064            0.013   
deepseek-reasoner            -0.014            -0.007           -0.013   
gpt-5-mini                    0.033            -0.054           -0.001   
gpt-5.1                       0.011            -0.045           -0.000   
mistral-large                 0.013            -0.015           -0.010   
mistral-small                 0.013            -0.068           -0.016   

bank                                                               FED  \
prompt            p4_analyst_json p5_analyst_fewshot p1_analyst_direct   
model                                                                    
claude-haiku               -0.005              0.014             0.024   
claude-sonnet               0.025             -0.013             0.020   
deepseek-chat               0.077             -0.061             0.077   
deepseek-reasoner           0.024              0.010             0.019   
gpt-5-mini                  0.033             -0.011             0.023   
gpt-5.1                     0.037             -0.003             0.017   
mistral-large               0.027             -0.015             0.001   
mistral-small               0.041              0.029             0.001   

bank                                                                  \
prompt            p2_analyst_rubric p3_analyst_rules p4_analyst_json   
model                                                                  
claude-haiku                  0.030           -0.006          -0.037   
claude-sonnet                 0.017           -0.022           0.013   
deepseek-chat                -0.166            0.058           0.074   
deepseek-reasoner             0.023           -0.007          -0.006   
gpt-5-mini                    0.006            0.010           0.004   
gpt-5.1                      -0.008            0.016          -0.000   
mistral-large                 0.007            0.013          -0.000   
mistral-small                -0.022            0.012           0.009   

bank                                  
prompt            p5_analyst_fewshot  
model                                 
claude-haiku                  -0.010  
claude-sonnet                 -0.029  
deepseek-chat                 -0.044  
deepseek-reasoner             -0.029  
gpt-5-mini                    -0.044  
gpt-5.1                       -0.025  
mistral-large                 -0.021  
mistral-small                 -0.001

C · relative skill (dataset-demeaned macro-F1)


bank,ECB,FED,ECB_minus_FED
model,,,
deepseek-reasoner,0.054,0.032,0.022
mistral-large,0.017,0.003,0.014
deepseek-chat,-0.058,-0.071,0.013
claude-sonnet,0.025,0.014,0.011
claude-haiku,0.009,0.009,0.000
gpt-5.1,0.014,0.025,-0.012
mistral-small,-0.042,-0.026,-0.016
gpt-5-mini,-0.018,0.014,-0.032


Best prompt per model × bank


,bank,model,prompt,macro_f1
0,ECB,deepseek-reasoner,p4_analyst_json,0.730
1,ECB,gpt-5.1,p4_analyst_json,0.702
2,ECB,claude-sonnet,p4_analyst_json,0.702
3,ECB,mistral-large,p4_analyst_json,0.695
4,ECB,claude-haiku,p1_analyst_direct,0.692
5,ECB,deepseek-chat,p4_analyst_json,0.671
6,ECB,gpt-5-mini,p1_analyst_direct,0.667
7,ECB,mistral-small,p4_analyst_json,0.650
8,FED,deepseek-reasoner,p2_analyst_rubric,0.676
9,FED,gpt-5.1,p1_analyst_direct,0.663
